# General Block Forward Example

This notebook shows callable objective and constraint blocks. The parameter data is sampled from a box region instead of loading `parameters.csv`.
        

In [ ]:
from pathlib import Path
import sys

import numpy as np

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'nlpoptnet').is_dir() and (path / 'notebooks').is_dir()
)
SRC = ROOT / 'nlpoptnet' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from nlpoptnet import NLPOptNet

import jax.numpy as jnp

In [ ]:
CONFIG = {
    'epochs': 3,
    'batch_size': 8,
    'learning_rate': 1e-3,
    'train_frac': 0.8,
    'hidden_size': 32,
    'hidden_layers': 2,
    'seed': 42,
    'dtype': 'float64',
    'print_every': 1,
    'verbose': True,
}

## Define a block-style problem
        

In [ ]:
model = NLPOptNet(config=CONFIG, name='notebook_general_block')
x = model.add_parameter(['x1', 'x2'])
y = model.add_variable(['y1', 'y2', 'y3'])

Q = jnp.diag(jnp.asarray([1.0, 1.2, 0.8], dtype=jnp.float64))
c = jnp.asarray([0.2, -0.1, 0.05], dtype=jnp.float64)


def objective(_params, vars):
    y_value = vars['y']
    return 0.5 * y_value @ Q @ y_value + c @ jnp.sin(y_value)


def equality_block(params, vars):
    x_value = params['x']
    y_value = vars['y']
    return jnp.asarray([
        y_value[0] + y_value[1] - x_value[0],
        y_value[1] - y_value[2] - x_value[1],
    ])


def inequality_block(_params, vars):
    y_value = vars['y']
    return jnp.asarray([
        y_value[0] ** 2 + y_value[2] ** 2 - 2.0,
        y_value[0] + 0.5 * y_value[1] - 1.5,
    ])


model.objective(objective)
model.constraints.equality.add(equality_block)
model.constraints.inequality.add(inequality_block)
model.constraints.box.add(var=y, lower=-2.0, upper=2.0)
model.box(lower=[-0.5, -0.5], upper=[0.5, 0.5], num_samples=32)
model.build()
result = model.optimize()
run_dir = Path(result['output_dir'])
run_dir
        

In [ ]:
model.predict([0.1, -0.2])
        